In [ ]:
# ================================
# Configuración inicial del proyecto
# Compatible con Colab y VS Code
# ================================

from pathlib import Path
import os
import sys

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    BASE_DIR = Path("/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto")

else:
    BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data"
FINAL_DIR = DATA_DIR / "final"
MODELS_DIR = BASE_DIR / "models"
DASHBOARD_DIR = BASE_DIR / "dashboard"

MODELS_DIR.mkdir(exist_ok=True)
DASHBOARD_DIR.mkdir(exist_ok=True)

print("Entorno Colab:", EN_COLAB)
print("BASE_DIR:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno Colab: True
BASE_DIR: /content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto


In [ ]:
# ================================
# Instalación de dependencias
# ================================

if EN_COLAB:
    !pip install -q pandas numpy scikit-learn statsmodels plotly xgboost shap joblib openpyxl

##  1 — Construcción de la base de modelado

A partir del dataset final consolidado (`df_mapa_sup_final`), se construye una base específica para modelado.

El objetivo es trabajar sobre una versión depurada del dataset que contenga únicamente observaciones válidas para el análisis, eliminando registros con valores faltantes en variables críticas.

En particular, se define como variable objetivo la tasa de delitos contra la propiedad cada 100.000 habitantes calculada con población anual (`tasa_delitos_propiedad_100k_v2`), ya que permite una mejor representación de la dinámica temporal del fenómeno.

Asimismo, se requiere la presencia de variables estructurales y temporales clave, como la densidad poblacional transformada (`log_densidad`) y el rezago de la variable objetivo (`tasa_lag1`), dado que estas resultan fundamentales para capturar tanto la heterogeneidad territorial como la persistencia temporal del delito.

Los valores faltantes en variables derivadas como tasas de variación o rezagos adicionales no son eliminados en esta etapa, ya que serán evaluados posteriormente durante el proceso de selección de variables.

Este procedimiento permite obtener una base consistente y adecuada para avanzar hacia la etapa de modelado.

In [ ]:


import pandas as pd
# carga dataset inicial
df_mapa_sup_final = pd.read_csv(
    FINAL_DIR / "dataset_mapa_sup_final.csv"
)


# Copia del dataset original
df_modelado = df_mapa_sup_final.copy()

print("Dimensión original:", df_modelado.shape)

# -----------------------------
# 1. Definir variable objetivo
# -----------------------------
target = "tasa_delitos_propiedad_100k_v2"

# -----------------------------
# 2. Definir variables críticas
# -----------------------------
variables_criticas = [
    target,
    "log_densidad",
    "tasa_lag1"
]

# -----------------------------
# 3. Eliminar nulos en variables clave
# -----------------------------
df_modelado = df_modelado.dropna(subset=variables_criticas).copy()

# -----------------------------
# 4. Ordenar dataset (importante para panel)
# -----------------------------
df_modelado = df_modelado.sort_values(
    by=["provincia_key", "departamento_key", "anio"]
).reset_index(drop=True)

# -----------------------------
# 5. Verificación
# -----------------------------
print("\nDimensión luego de limpieza:", df_modelado.shape)

print("\nNulos restantes:")
print(df_modelado.isna().sum().sort_values(ascending=False))

print("\nAños disponibles:")
print(sorted(df_modelado["anio"].unique()))

df_modelado.head()

Dimensión original: (4072, 34)

Dimensión luego de limpieza: (3542, 34)

Nulos restantes:
tasa_yoy                          6
provincia_id                      0
provincia_nombre                  0
departamento_id                   0
anio                              0
departamento_nombre               0
poblacion_2022                    0
tasa_delitos_propiedad_100k       0
tasa_delitos_propiedad_pct        0
delitos_propiedad_hechos          0
variacion_anual_salarios_pct      0
variacion_anual_cbt               0
variacion_empleo_const_pct        0
mes                               0
variacion_anual_ipim              0
jurisdiccion                      0
variacion_acceso_internet_pct     0
region                            0
Indice_IPC                        0
lat                               0
provincia_key                     0
lon                               0
poblacion                         0
Superficie en km2                 0
_merge                            0
departamen

,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct,region,...,_merge,flag_poblacion_faltante,tasa_delitos_propiedad_100k_v2,densidad_poblacion,tasa_lag1,Indice_IPC_lag1,variacion_anual_cbt_lag1,variacion_anual_salarios_lag1,tasa_yoy,log_densidad
0,6,buenos aires,6854,25 de mayo,2018,77,35411,217.446556,0.217447,Pampeana,...,both,False,208.689053,7.719345,239.078461,112.744967,26.77,27.47,-0.127111,2.165544
1,6,buenos aires,6854,25 de mayo,2019,207,35411,584.564118,0.584564,Pampeana,...,both,False,559.686359,7.737756,208.689053,151.744600,52.86,29.71,1.681915,2.167653
2,6,buenos aires,6854,25 de mayo,2020,129,35411,364.293581,0.364294,Pampeana,...,both,False,347.980902,7.755748,559.686359,232.926758,52.82,40.89,-0.378257,2.169710
3,6,buenos aires,6854,25 de mayo,2021,124,35411,350.173675,0.350174,Pampeana,...,both,False,333.728065,7.773532,347.980902,333.645717,39.14,32.99,-0.040959,2.171739
4,6,buenos aires,6854,25 de mayo,2022,169,35411,477.252831,0.477253,Pampeana,...,both,False,453.813104,7.791105,333.728065,499.073575,40.47,53.36,0.359829,2.173740


## 2 — Revisión inicial de variables para modelado

Una vez construida la base de modelado, se procede a realizar una primera selección de variables candidatas.

El objetivo de esta etapa es identificar aquellas variables numéricas que pueden aportar información relevante al modelo, excluyendo variables de identificación, geográficas o auxiliares que no resultan adecuadas para el entrenamiento.

Se consideran variables de tipo estructural, socioeconómico y temporal, incluyendo rezagos previamente construidos. Posteriormente, se analiza la relación entre estas variables mediante una matriz de correlación, con el fin de detectar posibles problemas de multicolinealidad y redundancia.

Este análisis constituye un paso previo fundamental para la construcción de modelos robustos e interpretables.

In [ ]:



# Variable objetivo
target = "tasa_delitos_propiedad_100k_v2"

# Variables candidatas
features = [
    "log_densidad",
    "Indice_IPC",
    "salarios" if "salarios" in df_modelado.columns else None,
    "variacion_anual_salarios_pct",
    "variacion_anual_cbt",
    "variacion_empleo_const_pct",
    "variacion_acceso_internet_pct",
    "variacion_anual_ipim",

    # Lags
    "tasa_lag1",
    "Indice_IPC_lag1",
    "variacion_anual_cbt_lag1",
    "variacion_anual_salarios_lag1",

    # Dinámica
    "tasa_yoy"
]

# Limpiar None por si alguna no existe
features = [f for f in features if f in df_modelado.columns]

# Dataset solo numérico para análisis
df_vars = df_modelado[[target] + features].copy()

print("Variables seleccionadas:")
print(df_vars.columns.tolist())

print("\nDimensión:", df_vars.shape)

Variables seleccionadas:
['tasa_delitos_propiedad_100k_v2', 'log_densidad', 'Indice_IPC', 'variacion_anual_salarios_pct', 'variacion_anual_cbt', 'variacion_empleo_const_pct', 'variacion_acceso_internet_pct', 'variacion_anual_ipim', 'tasa_lag1', 'Indice_IPC_lag1', 'variacion_anual_cbt_lag1', 'variacion_anual_salarios_lag1', 'tasa_yoy']

Dimensión: (3542, 13)


In [ ]:
from IPython.display import HTML

file_path = DASHBOARD_DIR / "correlaciones.html"

with open(file_path, "r", encoding="utf-8") as f:
    html_content = f.read()

HTML(html_content)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
corr = df_vars.corr()
corr_target = corr[target].sort_values(ascending=False)
print("Correlación con el target:\n")
print(corr_target)

Correlación con el target:

tasa_delitos_propiedad_100k_v2    1.000000
tasa_lag1                         0.923929
log_densidad                      0.471335
variacion_anual_salarios_pct      0.142819
variacion_anual_salarios_lag1     0.132197
Indice_IPC_lag1                   0.127162
variacion_anual_cbt_lag1          0.125690
Indice_IPC                        0.125563
variacion_anual_cbt               0.110556
variacion_anual_ipim              0.080533
tasa_yoy                          0.025346
variacion_empleo_const_pct        0.013353
variacion_acceso_internet_pct    -0.002024
Name: tasa_delitos_propiedad_100k_v2, dtype: float64


## 3 — Reducción de multicolinealidad y selección de variables

A partir del análisis de correlación realizado en la etapa anterior, se identificó la presencia de multicolinealidad significativa entre diversas variables económicas, particularmente entre indicadores de precios, ingresos y actividad.

Dado que la inclusión simultánea de variables altamente correlacionadas puede generar inestabilidad en los modelos e impedir una correcta interpretación de los coeficientes, se procede a seleccionar un subconjunto representativo de variables.

En este sentido, se adopta un criterio de selección basado en la representatividad económica de cada indicador, evitando redundancias. Asimismo, se conservan variables estructurales y temporales clave que permiten capturar la heterogeneidad territorial y la dinámica del fenómeno.

Como resultado, se define un conjunto reducido de variables para el modelado, priorizando interpretabilidad y estabilidad estadística.

In [ ]:


target = "tasa_delitos_propiedad_100k_v2"

features_finales = [
    # Estructurales
    "log_densidad",

    # Económicas (seleccionadas)
    "variacion_anual_cbt",
    "variacion_anual_salarios_pct",
    "variacion_empleo_const_pct",
    "variacion_acceso_internet_pct",

    # Temporal clave
    "tasa_lag1"
]

# Dataset final de modelado
df_final = df_modelado[[target] + features_finales].copy()

print("Variables finales:")
print(df_final.columns.tolist())

print("\nDimensión:", df_final.shape)

df_final.head()

Variables finales:
['tasa_delitos_propiedad_100k_v2', 'log_densidad', 'variacion_anual_cbt', 'variacion_anual_salarios_pct', 'variacion_empleo_const_pct', 'variacion_acceso_internet_pct', 'tasa_lag1']

Dimensión: (3542, 7)


,tasa_delitos_propiedad_100k_v2,log_densidad,variacion_anual_cbt,variacion_anual_salarios_pct,variacion_empleo_const_pct,variacion_acceso_internet_pct,tasa_lag1
0,208.689053,2.165544,52.86,29.71,2.804384,-0.084572,239.078461
1,559.686359,2.167653,52.82,40.89,-4.095870,3.259377,208.689053
2,347.980902,2.169710,39.14,32.99,-22.513076,-0.906055,559.686359
3,333.728065,2.171739,40.47,53.36,10.894769,5.042552,347.980902
4,453.813104,2.173740,100.29,90.36,16.258769,9.714659,333.728065


In [ ]:
print("\nDimensión:", df_final.shape)

print("\nNulos restantes:")
print(df_final.isna().sum().sort_values(ascending=False))


Dimensión: (3542, 7)

Nulos restantes:
tasa_delitos_propiedad_100k_v2    0
log_densidad                      0
variacion_anual_cbt               0
variacion_anual_salarios_pct      0
variacion_empleo_const_pct        0
variacion_acceso_internet_pct     0
tasa_lag1                         0
dtype: int64


## 4 — Modelo baseline: Regresión Lineal

Como primer enfoque, se construye un modelo de regresión lineal múltiple utilizando el conjunto de variables seleccionado en la etapa anterior.

El objetivo de este modelo es establecer una línea base de referencia que permita evaluar la capacidad explicativa de las variables consideradas, así como analizar la dirección e intensidad de las relaciones entre estas y la variable objetivo.

Si bien este modelo no contempla efectos específicos por territorio ni dinámicas más complejas, resulta útil para obtener una primera aproximación al comportamiento del fenómeno y validar la consistencia del dataset construido.

Los resultados obtenidos serán posteriormente comparados con modelos más sofisticados, incluyendo modelos de datos de panel y técnicas de machine learning.

In [ ]:


from sklearn.model_selection import train_test_split

# Variables
X = df_final.drop(columns=["tasa_delitos_propiedad_100k_v2"])
y = df_final["tasa_delitos_propiedad_100k_v2"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (3542, 6)
Shape y: (3542,)


In [ ]:
# Split temporal
train = df_modelado[df_modelado["anio"] <= 2022]
test  = df_modelado[df_modelado["anio"] > 2022]

# Aplicar mismas columnas
X_train = train[X.columns]
y_train = train["tasa_delitos_propiedad_100k_v2"]

X_test = test[X.columns]
y_test = test["tasa_delitos_propiedad_100k_v2"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2500, 6)
Test: (1042, 6)


In [ ]:
from sklearn.linear_model import LinearRegression
# Entrenar modelo
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Prediccion y evaluación
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 453.92563695033175
RMSE: 559.6083642131001
R2: 0.7835652412100025


In [ ]:
# Ver coeficientes para interpretar importancia de variables
coeficientes = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": model.coef_
}).sort_values(by="coeficiente", key=abs, ascending=False)

coeficientes

,variable,coeficiente
0,log_densidad,22.941036
3,variacion_empleo_const_pct,13.414495
4,variacion_acceso_internet_pct,-5.709315
1,variacion_anual_cbt,-1.140648
2,variacion_anual_salarios_pct,-1.042326
5,tasa_lag1,0.902959


## Interpretación — Modelo baseline: Regresión Lineal

El modelo de regresión lineal múltiple estimado constituye una primera aproximación al análisis del fenómeno, permitiendo evaluar la relación entre la tasa de delitos contra la propiedad y un conjunto de variables estructurales, económicas y temporales.

### Calidad del ajuste

El modelo presenta un **R² de 0.78**, lo que indica que aproximadamente el 78% de la variabilidad de la tasa de delitos es explicada por las variables incluidas. Este nivel de ajuste es elevado para un modelo base, lo que sugiere que el conjunto de variables seleccionado captura una parte significativa del comportamiento del fenómeno.

Los valores de **MAE (≈ 454)** y **RMSE (≈ 560)** reflejan un nivel de error considerable en términos absolutos, lo cual es esperable dada la alta dispersión observada en la variable objetivo durante el análisis exploratorio.

### Persistencia temporal del delito

La variable más relevante del modelo es el rezago de la tasa (`tasa_lag1`), con un coeficiente cercano a **0.90**, lo que indica una fuerte relación entre el nivel de delitos de un período y el del período anterior.

Este resultado sugiere que el delito presenta una marcada **inercia temporal**, es decir, los niveles actuales están fuertemente condicionados por los niveles históricos recientes.

### Efectos estructurales

La variable `log_densidad` presenta un coeficiente positivo significativo, indicando que, a mayor densidad poblacional, mayor es la tasa de delitos contra la propiedad.

Este resultado es consistente con la hipótesis de que los entornos urbanos, caracterizados por mayor concentración de población, presentan mayores oportunidades para la ocurrencia de este tipo de delitos.

### Variables económicas

Las variables económicas incluidas muestran efectos más moderados:

- La **variación de la CBT** y la **variación de salarios** presentan coeficientes negativos, lo que sugiere que mejoras en las condiciones económicas podrían estar asociadas con una reducción en la tasa de delitos.
- La **variación del empleo en la construcción** presenta un coeficiente positivo, lo cual puede interpretarse como un indicador de mayor actividad económica o dinamismo urbano, potencialmente asociado a mayores oportunidades delictivas.
- La **variación en el acceso a internet** muestra un efecto negativo, que podría estar capturando aspectos vinculados al desarrollo o nivel socioeconómico de los territorios.

### Limitaciones del modelo

A pesar de sus resultados, este modelo presenta limitaciones importantes:

- No controla por diferencias estructurales entre departamentos (heterogeneidad no observada)
- No incorpora efectos específicos de cada año (shocks macroeconómicos o coyunturales)
- Puede estar sesgado por variables omitidas que varían entre territorios

Por lo tanto, si bien el modelo permite una primera interpretación de las relaciones entre variables, sus resultados deben considerarse preliminares.


## 5 — Modelo de datos de panel con efectos fijos

Con el objetivo de mejorar la estimación del modelo y controlar por factores no observados, se implementa un modelo de regresión con efectos fijos.

Este enfoque permite capturar:

- diferencias estructurales entre departamentos que permanecen constantes en el tiempo (efectos fijos por unidad territorial)
- shocks o cambios comunes a todos los departamentos en un mismo año (efectos fijos temporales)

De esta manera, se obtiene una estimación más robusta del efecto de las variables explicativas, aislando el impacto de factores no observados que podrían sesgar los resultados en un modelo de regresión tradicional.

Este tipo de modelo resulta especialmente adecuado para datos de tipo panel, como el presente caso, donde se observan múltiples unidades territoriales a lo largo del tiempo.

In [ ]:
import statsmodels.formula.api as smf

# Dataset de entrenamiento (el mismo que usaste antes)
df_train = train.copy()

# Fórmula del modelo
formula = """
tasa_delitos_propiedad_100k_v2 ~
log_densidad +
variacion_anual_cbt +
variacion_anual_salarios_pct +
variacion_empleo_const_pct +
variacion_acceso_internet_pct +
tasa_lag1 +
C(departamento_key) +
C(anio)
"""

# Entrenamiento
modelo_fe = smf.ols(formula, data=df_train).fit()

# Resumen
print(modelo_fe.summary())

                                  OLS Regression Results                                  
Dep. Variable:     tasa_delitos_propiedad_100k_v2   R-squared:                       0.877
Model:                                        OLS   Adj. R-squared:                  0.852
Method:                             Least Squares   F-statistic:                     34.93
Date:                            Thu, 28 May 2026   Prob (F-statistic):               0.00
Time:                                    02:18:10   Log-Likelihood:                -18058.
No. Observations:                            2500   AIC:                         3.697e+04
Df Residuals:                                2074   BIC:                         3.945e+04
Df Model:                                     425                                         
Covariance Type:                        nonrobust                                         
                                                           coef    std err          t     

## Interpretación — Modelo de datos de panel con efectos fijos

El modelo estimado incorpora efectos fijos por departamento y por año, lo que permite controlar por características no observadas que permanecen constantes en el tiempo (heterogeneidad territorial) y por shocks comunes a todos los departamentos en un mismo período (efectos temporales).

### Calidad del ajuste

El modelo presenta un **R² de 0.877** y un **R² ajustado de 0.852**, lo que indica una mejora respecto al modelo baseline. Esto sugiere que, al incorporar efectos fijos, se logra capturar una mayor proporción de la variabilidad de la tasa de delitos, especialmente aquella asociada a diferencias estructurales entre departamentos.

### Efectos fijos

La inclusión de una gran cantidad de variables dummy (`C(departamento_key)` y `C(anio)`) permite modelar:

- diferencias persistentes entre departamentos (por ejemplo, características socioeconómicas, institucionales o urbanas no observadas)
- variaciones comunes en el tiempo (como cambios macroeconómicos o eventos como la pandemia)

Los coeficientes asociados a los departamentos no deben interpretarse individualmente, sino como ajustes que capturan estas diferencias estructurales.

### Variables explicativas

A diferencia del modelo baseline, donde las variables económicas tenían una interpretación más directa, en este modelo los coeficientes deben leerse como:

> el efecto de cada variable **dentro de un mismo departamento a lo largo del tiempo**, manteniendo constantes las diferencias estructurales entre territorios y los efectos específicos de cada año.

Esto implica que:

- la **tasa rezagada (`tasa_lag1`)** continúa capturando la persistencia temporal del delito
- las variables económicas reflejan ahora su impacto **neto**, descontando efectos territoriales fijos
- la **densidad poblacional (`log_densidad`)** puede perder parte de su variabilidad explicativa si no cambia significativamente dentro de cada departamento a lo largo del tiempo

### Multicolinealidad y advertencia técnica

El modelo arroja una advertencia relacionada con un **autovalor extremadamente pequeño**, lo que indica posibles problemas de:

- multicolinealidad residual
- o redundancia en la matriz de diseño (producto de la gran cantidad de variables dummy)

Este fenómeno es común en modelos con muchos efectos fijos y no invalida el análisis, pero implica que:

- algunos coeficientes pueden ser inestables
- la interpretación debe centrarse en patrones generales más que en valores puntuales

### Conclusión

El modelo de efectos fijos representa una mejora sustancial respecto al baseline, ya que permite aislar el impacto de las variables explicativas controlando por factores no observados.

Los resultados refuerzan la idea de que:

- el delito presenta una fuerte **persistencia temporal**
- existen **diferencias estructurales significativas entre departamentos**
- las variables económicas tienen un efecto **acotado y condicionado por el contexto territorial**

Este enfoque proporciona una base más sólida para el análisis explicativo del fenómeno y constituye un paso clave en la validación del modelo.

## 6 — Evaluación del modelo de efectos fijos

Si bien el modelo de efectos fijos fue planteado con un enfoque principalmente explicativo, resulta útil evaluar su desempeño predictivo sobre datos no utilizados en el entrenamiento.

Para ello, se generan predicciones sobre el conjunto de test (años más recientes) y se calculan métricas estándar como MAE, RMSE y R².

Este análisis permite comparar el rendimiento del modelo de panel con el modelo baseline previamente estimado, y evaluar en qué medida la incorporación de efectos fijos mejora la capacidad predictiva del modelo.

In [ ]:
# Predicción usando el modelo de panel
y_pred_fe = modelo_fe.predict(test)

PatsyError: predict requires that you use a DataFrame when predicting from a model
that was created using the formula api.

The original error message returned by patsy is:
Error converting data to categorical: observation with value 'atreuco' does not match any of the expected levels (expected: ['12 de octubre', '1o de mayo', ..., 'zarate', 'zonda'])
    tasa_delitos_propiedad_100k_v2 ~ log_densidad + variacion_anual_cbt + variacion_anual_salarios_pct + variacion_empleo_const_pct + variacion_acceso_internet_pct + tasa_lag1 + C(departamento_key) + C(anio)
                                                                                                                                                                                  ^^^^^^^^^^^^^^^^^^^

### Ajuste técnico para la predicción del modelo de efectos fijos

Al intentar generar predicciones sobre el conjunto de test, se detectó que algunos departamentos presentes en los años de evaluación no habían sido observados en el conjunto de entrenamiento.

Dado que el modelo de efectos fijos incorpora variables categóricas por departamento (`C(departamento_key)`), solo puede predecir correctamente para niveles ya conocidos durante el ajuste.

Por este motivo, para evaluar el desempeño predictivo del modelo de panel, se restringe el conjunto de test a aquellos departamentos que también están presentes en el conjunto de entrenamiento. Esta decisión permite realizar una comparación válida dentro de las unidades efectivamente observadas por el modelo.

In [ ]:
# Departamentos presentes en train
dept_train = set(train["departamento_key"].unique())

# Filtrar test solo con departamentos ya vistos en train
test_fe = test[test["departamento_key"].isin(dept_train)].copy()

print("Test original:", test.shape)
print("Test filtrado para FE:", test_fe.shape)

# Verificar si quedó alguno raro
dept_test_fe = set(test_fe["departamento_key"].unique())
print("Departamentos en test_fe que no están en train:", dept_test_fe - dept_train)

Test original: (1042, 34)
Test filtrado para FE: (1001, 34)
Departamentos en test_fe que no están en train: set()


In [ ]:
y_test_fe = test_fe["tasa_delitos_propiedad_100k_v2"]
y_pred_fe = modelo_fe.predict(test_fe)

PatsyError: predict requires that you use a DataFrame when predicting from a model
that was created using the formula api.

The original error message returned by patsy is:
Error converting data to categorical: observation with value 2023 does not match any of the expected levels (expected: [2018, 2019, ..., 2021, 2022])
    tasa_delitos_propiedad_100k_v2 ~ log_densidad + variacion_anual_cbt + variacion_anual_salarios_pct + variacion_empleo_const_pct + variacion_acceso_internet_pct + tasa_lag1 + C(departamento_key) + C(anio)
                                                                                                                                                                                                        ^^^^^^^

##  6 — Modelo con efectos fijos por departamento para evaluación fuera de muestra

Dado que el modelo de efectos fijos con controles por departamento y por año fue diseñado con fines principalmente explicativos, su uso para predicción fuera de muestra presenta una limitación metodológica: no puede generar predicciones para años no observados durante el entrenamiento.

Por este motivo, se estima una especificación alternativa con efectos fijos únicamente por departamento. Esta versión conserva el control por heterogeneidad territorial no observada, pero permite evaluar el desempeño predictivo sobre años posteriores, siempre que las unidades territoriales hayan sido observadas previamente en el conjunto de entrenamiento.

Esta estrategia permite comparar el rendimiento predictivo del modelo frente al baseline sin perder completamente la estructura de panel del problema.

In [ ]:
import statsmodels.formula.api as smf

# Modelo FE solo por departamento
formula_fe_dep = """
tasa_delitos_propiedad_100k_v2 ~
log_densidad +
variacion_anual_cbt +
variacion_anual_salarios_pct +
variacion_empleo_const_pct +
variacion_acceso_internet_pct +
tasa_lag1 +
C(departamento_key)
"""

modelo_fe_dep = smf.ols(formula_fe_dep, data=train).fit()

print(modelo_fe_dep.summary())

                                  OLS Regression Results                                  
Dep. Variable:     tasa_delitos_propiedad_100k_v2   R-squared:                       0.870
Model:                                        OLS   Adj. R-squared:                  0.844
Method:                             Least Squares   F-statistic:                     32.83
Date:                            Thu, 28 May 2026   Prob (F-statistic):               0.00
Time:                                    02:19:07   Log-Likelihood:                -18129.
No. Observations:                            2500   AIC:                         3.711e+04
Df Residuals:                                2075   BIC:                         3.958e+04
Df Model:                                     424                                         
Covariance Type:                        nonrobust                                         
                                                           coef    std err          t     

In [ ]:
# Filtrar test para departamentos vistos en train
dept_train = set(train["departamento_key"].unique())
test_fe_dep = test[test["departamento_key"].isin(dept_train)].copy()

y_test_fe_dep = test_fe_dep["tasa_delitos_propiedad_100k_v2"]
y_pred_fe_dep = modelo_fe_dep.predict(test_fe_dep)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_fe_dep = mean_absolute_error(y_test_fe_dep, y_pred_fe_dep)
rmse_fe_dep = np.sqrt(mean_squared_error(y_test_fe_dep, y_pred_fe_dep))
r2_fe_dep = r2_score(y_test_fe_dep, y_pred_fe_dep)

print("Modelo FE por departamento")
print("MAE:", mae_fe_dep)
print("RMSE:", rmse_fe_dep)
print("R2:", r2_fe_dep)

Modelo FE por departamento
MAE: 377.5990519933308
RMSE: 496.36167691931365
R2: 0.8325753889631788


## Interpretación — Modelo de efectos fijos por departamento

El modelo estimado incorpora efectos fijos por departamento, lo que permite controlar por características estructurales no observadas que diferencian a las distintas unidades territoriales.

En términos de ajuste, el modelo presenta un R² de aproximadamente 0.87 sobre el conjunto de entrenamiento, lo que indica una alta capacidad explicativa. Este valor es comparable al obtenido en el modelo de efectos fijos completo, lo que sugiere que gran parte de la variabilidad del delito se explica por diferencias estructurales entre departamentos más que por efectos temporales comunes.

Al evaluar el modelo sobre datos no utilizados en el entrenamiento (años posteriores), se obtiene un R² cercano a 0.83, junto con valores de error (MAE ≈ 378 y RMSE ≈ 496) consistentes con la escala de la variable objetivo. Esto indica que el modelo mantiene una buena capacidad predictiva fuera de muestra, sin evidencias fuertes de sobreajuste.

A diferencia del modelo que incorpora efectos por año, esta especificación permite realizar predicciones sobre períodos futuros, manteniendo el control por heterogeneidad territorial. Esto resulta especialmente relevante en contextos donde se busca evaluar el comportamiento del modelo en escenarios reales.

En este sentido, el modelo logra un equilibrio entre capacidad explicativa y aplicabilidad predictiva, constituyéndose como una herramienta adecuada tanto para el análisis del fenómeno como para su modelado.

Estos resultados refuerzan la hipótesis de que el delito contra la propiedad presenta una fuerte componente territorial, con diferencias persistentes entre regiones, mientras que las variables económicas contribuyen a explicar parcialmente su dinámica.

## 7 — Modelo predictivo: Random Forest

Con el objetivo de mejorar la capacidad predictiva del modelo, se implementa un algoritmo de Random Forest.

Este enfoque, basado en ensamblado de árboles de decisión, permite capturar relaciones no lineales e interacciones entre variables sin necesidad de especificarlas explícitamente.

A diferencia de los modelos de regresión tradicionales, el Random Forest no requiere supuestos estrictos sobre la distribución de los datos y es robusto frente a problemas como la multicolinealidad.

El desempeño del modelo será evaluado sobre el conjunto de test, permitiendo comparar su capacidad predictiva frente a los modelos previamente desarrollados.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Entrenar modelo Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [ ]:
# Prediccion
y_pred_rf = rf.predict(X_test)

In [ ]:
# Metricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

Random Forest
MAE: 269.3560900643051
RMSE: 386.0521009997988
R2: 0.8969968621708634


In [ ]:
import pandas as pd

# Importancia de variables en Random Forest

importancia = pd.DataFrame({
    "variable": X_train.columns,
    "importancia": rf.feature_importances_
}).sort_values(by="importancia", ascending=False)

importancia

,variable,importancia
5,tasa_lag1,0.888255
0,log_densidad,0.035218
4,variacion_acceso_internet_pct,0.025575
3,variacion_empleo_const_pct,0.022282
1,variacion_anual_cbt,0.021331
2,variacion_anual_salarios_pct,0.007338


## Interpretación — Modelo Random Forest

El modelo de Random Forest presenta una mejora en el desempeño predictivo, alcanzando un R² cercano a 0.90 y reduciendo significativamente los errores de predicción en comparación con los modelos lineales y de panel.

Estos resultados indican que la relación entre las variables explicativas y la tasa de delitos contra la propiedad no es estrictamente lineal, sino que involucra interacciones y comportamientos más complejos que son mejor capturados por modelos no paramétricos.

El análisis de importancia de variables muestra que la tasa rezagada (`tasa_lag1`) continúa siendo el predictor más relevante, lo que refuerza la evidencia de una fuerte persistencia temporal del delito. Asimismo, la densidad poblacional mantiene un rol importante, confirmando la influencia de factores estructurales territoriales.

Las variables económicas, si bien contribuyen al modelo, presentan una importancia relativa menor, lo que sugiere que el fenómeno delictivo no puede ser explicado únicamente por condiciones económicas, sino que responde a una combinación de factores.


##  8 — Modelo predictivo: XGBoost


In [ ]:


from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=75,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost")
print("MAE:", mae_xgb)
print("RMSE:", rmse_xgb)
print("R2:", r2_xgb)

importancia_xgb = pd.DataFrame({
    "variable": X_train.columns,
    "importancia": xgb.feature_importances_
}).sort_values(by="importancia", ascending=False)

importancia_xgb

XGBoost
MAE: 267.37988584972095
RMSE: 381.7126896438569
R2: 0.899299457541464


,variable,importancia
5,tasa_lag1,0.727490
1,variacion_anual_cbt,0.096562
2,variacion_anual_salarios_pct,0.078496
0,log_densidad,0.043939
3,variacion_empleo_const_pct,0.026957
4,variacion_acceso_internet_pct,0.026556


## 9 — Comparación entre modelos

In [ ]:
import plotly.graph_objects as go

modelos = ["Baseline", "FE Departamento", "Random Forest", "XGBoost"]

fig = go.Figure()

# Barras MAE
fig.add_trace(go.Bar(
    x=modelos,
    y=[mae, mae_fe_dep, mae_rf, mae_xgb],
    name="MAE"
))

# Barras RMSE
fig.add_trace(go.Bar(
    x=modelos,
    y=[rmse, rmse_fe_dep, rmse_rf, rmse_xgb],
    name="RMSE"
))

# Línea R2 (eje secundario)
fig.add_trace(go.Scatter(
    x=modelos,
    y=[r2, r2_fe_dep, r2_rf, r2_xgb],
    name="R²",
    yaxis="y2",
    mode="lines+markers+text",
    text=[round(r2, 3), round(r2_fe_dep, 3), round(r2_rf, 3), round(r2_xgb, 3)],
    textposition="top center",
    line=dict(width=3)
))

# Layout
fig.update_layout(
    template="plotly_dark",
    title={
        "text": "Comparación de métricas entre modelos",
        "x": 0.5
    },
    barmode="group",

    xaxis=dict(title="Modelo"),

    yaxis=dict(
        title="MAE / RMSE"
    ),

    yaxis2=dict(
        title="R²",
        overlaying="y",
        side="right",
        range=[0, 1]
    ),

    legend_title="Métrica"
)

fig.show()

# Guardar grafico en dashboard
fig.write_html(
    DASHBOARD_DIR / "comparación_modelos.html"
)

In [ ]:
# Tabla de métricas
metricas_modelos = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Efectos Fijos", "Random Forest", "XGBoost"],
    "MAE": [mae, mae_fe_dep, mae_rf, mae_xgb],
    "RMSE": [rmse, rmse_fe_dep, rmse_rf, rmse_xgb],
    "R2": [r2, r2_fe_dep, r2_rf, r2_xgb]
})

metricas_modelos

,Modelo,MAE,RMSE,R2
0,Regresión Lineal,453.925637,559.608364,0.783565
1,Efectos Fijos,377.599052,496.361677,0.832575
2,Random Forest,269.356090,386.052101,0.896997
3,XGBoost,267.379886,381.712690,0.899299


In [ ]:
import plotly.graph_objects as go

# Límites para la línea ideal
min_val = min(y_test.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_rf.max())

fig = go.Figure()

# Puntos reales vs predichos
fig.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_rf,
    mode="markers",
    name="Observaciones",
    opacity=0.45
))

# Línea ideal y = x
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    name="Predicción perfecta",
    line=dict(width=3)
))

fig.update_layout(
    template="plotly_dark",
    title={
        "text": "Random Forest — Valores reales vs predichos",
        "x": 0.5
    },
    xaxis_title="Valor real",
    yaxis_title="Valor predicho"
)

fig.show()

In [ ]:
import plotly.express as px

# Ordenar de mayor a menor importancia
importancia_plot = importancia.sort_values(by="importancia", ascending=True)

fig = px.bar(
    importancia_plot,
    x="importancia",
    y="variable",
    orientation="h",
    template="plotly_dark",
    title="Importancia de variables — Random Forest"
)

fig.update_layout(
    title={"x": 0.5},
    xaxis_title="Importancia",
    yaxis_title="Variable"
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Errores
errores_baseline = y_test - y_pred
errores_rf = y_test - y_pred_rf

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=errores_baseline,
    name="Baseline",
    opacity=0.6,
    nbinsx=50
))

fig.add_trace(go.Histogram(
    x=errores_rf,
    name="Random Forest",
    opacity=0.6,
    nbinsx=50
))

fig.update_layout(
    template="plotly_dark",
    title={
        "text": "Distribución de errores: Baseline vs Random Forest",
        "x": 0.5
    },
    xaxis_title="Error (real - predicho)",
    yaxis_title="Frecuencia",
    barmode="overlay",
    legend_title="Modelo"
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Residuos
residuos_rf = y_test - y_pred_rf

fig = go.Figure()

# Puntos
fig.add_trace(go.Scatter(
    x=y_pred_rf,
    y=residuos_rf,
    mode="markers",
    name="Residuos",
    opacity=0.5
))

# Línea horizontal en 0 (error perfecto)
fig.add_trace(go.Scatter(
    x=[y_pred_rf.min(), y_pred_rf.max()],
    y=[0, 0],
    mode="lines",
    name="Error = 0",
    line=dict(width=3)
))

fig.update_layout(
    template="plotly_dark",
    title={
        "text": "Residuos vs valores predichos — Random Forest",
        "x": 0.5
    },
    xaxis_title="Valor predicho",
    yaxis_title="Residuo (real - predicho)"
)

fig.show()

## Interpretación integral de los resultados y visualizaciones

El conjunto de gráficos construidos permite realizar una evaluación completa del desempeño de los modelos desarrollados, abarcando tanto el análisis cuantitativo como cualitativo del comportamiento predictivo.

---

## 1. Comparación de métricas entre modelos

El gráfico de métricas muestra una mejora progresiva entre los distintos enfoques:

- El modelo **baseline** presenta el menor desempeño.
- El modelo de **efectos fijos por departamento** mejora significativamente, evidenciando la importancia de controlar la heterogeneidad territorial.
- El modelo **Random Forest** y **XGBoost** alcanzan el mejor rendimiento en todas las métricas, siendo **XGBoost** ligeramente superior:
  - Mayor R² (~0.90 para ambos, con XGBoost un poco por encima)
  - Menor MAE y RMSE para XGBoost

Esto indica que los modelos lineales capturan parte importante del fenómeno, pero la incorporación de relaciones no lineales mejora notablemente la capacidad predictiva.

---

## 2. Valores reales vs predichos (Ejemplo: Random Forest)

El gráfico de dispersión (mostrado para Random Forest, un modelo de alto rendimiento) muestra una clara alineación de los puntos en torno a la diagonal (línea de predicción perfecta).

**Interpretación:**

- Existe una buena correspondencia entre valores reales y predichos
- El modelo captura correctamente la tendencia general
- Se observan mayores desviaciones en valores altos → presencia de mayor dificultad en extremos (outliers)

---

## 3. Importancia de variables

El gráfico muestra una jerarquía muy marcada donde:

- El delito presenta una **fuerte persistencia temporal**
- La **estructura territorial (densidad)** es el segundo factor clave
- Las variables económicas tienen un rol **complementario, no dominante**

Esto refuerza la hipótesis de un fenómeno **multicausal pero fuertemente estructurado**.

---

## 4. Distribución de errores

El histograma comparativo muestra que los modelos avanzados (Random Forest y XGBoost) no solo mejoran el error promedio, sino también:
- la **estabilidad del modelo**
- la **robustez frente a valores extremos**

---

## 5. Residuos vs valores predichos

El gráfico presenta una dispersión relativamente aleatoria alrededor de 0, sin patrones sistemáticos claros y con un leve aumento de dispersión en valores altos.

**Interpretación:**

- No se detectan sesgos estructurales fuertes
- El modelo captura adecuadamente la relación general
- Existe cierta **heterocedasticidad leve**, típica en variables con alta variabilidad

## Conclusión integradora

El análisis conjunto de los gráficos permite concluir que:

- El fenómeno delictivo presenta una **fuerte persistencia temporal** (confirmado por la dominancia de `tasa_lag1`)
- Existe una **importante heterogeneidad territorial**, capturada por la densidad poblacional
- Las variables económicas influyen, pero **no explican por sí solas el fenómeno**

Desde el punto de vista metodológico:

- Los modelos lineales son útiles para interpretación
- Los modelos de panel permiten controlar factores estructurales
- El modelo Random Forest ofrece la mejor capacidad predictiva al capturar relaciones no lineales

En conjunto, los resultados sugieren que el delito contra la propiedad en Argentina es un fenómeno:

- dinámico
- territorialmente desigual
- parcialmente influenciado por condiciones económicas
- mejor modelado mediante enfoques híbridos que combinen interpretación y capacidad predictiva

## 10 — SHAP para explicar el modelo ganador

In [ ]:
mejor_modelo_nombre = metricas_modelos.sort_values("RMSE").iloc[0]["Modelo"]
mejor_modelo_nombre

'XGBoost'

In [ ]:


import shap
import joblib
import pandas as pd


if mejor_modelo_nombre == "XGBoost":
    modelo_ganador = xgb
elif mejor_modelo_nombre == "Random Forest":
    modelo_ganador = rf
else:
    modelo_ganador = None

if modelo_ganador is not None:
    # Muestra para que SHAP sea liviano
    X_shap = X_test.sample(
        n=min(500, len(X_test)),
        random_state=42
    )

    explainer = shap.TreeExplainer(modelo_ganador)
    shap_values = explainer.shap_values(X_shap)

    joblib.dump(
    modelo_ganador,
    MODELS_DIR / "modelo_ganador.pkl"
  )

    joblib.dump(
    mejor_modelo_nombre,
    MODELS_DIR / "modelo_ganador_nombre.pkl"
  )

    joblib.dump(
    X_shap,
    MODELS_DIR / "X_shap_sample.pkl"
  )

    joblib.dump(
    shap_values,
    MODELS_DIR / "shap_values.pkl"
)
    print("SHAP guardado para:", mejor_modelo_nombre)
else:
    print("El modelo ganador no es de tipo árbol compatible con SHAP en este bloque.")

SHAP guardado para: XGBoost


In [ ]:
import joblib

# ================================
# Guardado de modelos y artefactos
# ================================

# Modelos
joblib.dump(
    rf,
    MODELS_DIR / "random_forest.pkl"
)

joblib.dump(
    xgb,
    MODELS_DIR / "xgboost.pkl"
)

# Features utilizadas
joblib.dump(
    X.columns.tolist(),
    MODELS_DIR / "features_modelo.pkl"
)

# Métricas
metricas_modelos.to_csv(
    MODELS_DIR / "metricas_modelos.csv",
    index=False
)

# Importancias
importancia.to_csv(
    MODELS_DIR / "importancias_rf.csv",
    index=False
)

importancia_xgb.to_csv(
    MODELS_DIR / "importancias_xgb.csv",
    index=False
)

print("Artefactos guardados correctamente en /models")

Artefactos guardados correctamente en /models


## 11 — Rolling / Expanding temporal

In [ ]:


from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

resultados_temporales = []

anios_validacion = sorted(df_modelado["anio"].unique())

# Se empieza desde 2020 para tener años previos suficientes
for anio_test in anios_validacion:
    if anio_test <= 2020:
        continue

    train_temp = df_modelado[df_modelado["anio"] < anio_test].copy()
    test_temp = df_modelado[df_modelado["anio"] == anio_test].copy()

    train_temp = train_temp.dropna(subset=[target] + features_finales)
    test_temp = test_temp.dropna(subset=[target] + features_finales)

    if train_temp.empty or test_temp.empty:
        continue

    X_train_temp = train_temp[features_finales]
    y_train_temp = train_temp[target]

    X_test_temp = test_temp[features_finales]
    y_test_temp = test_temp[target]

    modelos_temp = {
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    for nombre, modelo_temp in modelos_temp.items():
        modelo_temp.fit(X_train_temp, y_train_temp)
        pred_temp = modelo_temp.predict(X_test_temp)

        resultados_temporales.append({
            "anio_test": anio_test,
            "modelo": nombre,
            "MAE": mean_absolute_error(y_test_temp, pred_temp),
            "RMSE": np.sqrt(mean_squared_error(y_test_temp, pred_temp)),
            "R2": r2_score(y_test_temp, pred_temp)
        })

rolling_resultados = pd.DataFrame(resultados_temporales)
rolling_resultados

,anio_test,modelo,MAE,RMSE,R2
0,2021,Random Forest,316.343612,459.931796,0.734767
1,2021,XGBoost,297.299406,454.171365,0.741369
2,2022,Random Forest,250.236229,328.439745,0.904819
3,2022,XGBoost,249.053202,324.203244,0.907259
4,2023,Random Forest,250.891438,352.124531,0.911770
5,2023,XGBoost,272.241722,406.051548,0.882677
6,2024,Random Forest,276.982747,387.747597,0.898685
7,2024,XGBoost,274.605965,389.877181,0.897570


In [ ]:
# Guardarlo
rolling_resultados.to_csv(
    MODELS_DIR / "rolling_temporal_resultados.csv",
    index=False
)

In [ ]:
# ================================
# Robustez 1 — Modelos sin tasa_lag1
# ================================

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Features sin tasa_lag1
features_sin_lag = [col for col in features_finales if col != "tasa_lag1"]

X_train_sin_lag = train[features_sin_lag]
X_test_sin_lag = test[features_sin_lag]

# Random Forest sin lag
rf_sin_lag = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_sin_lag.fit(X_train_sin_lag, y_train)
y_pred_rf_sin_lag = rf_sin_lag.predict(X_test_sin_lag)

# XGBoost sin lag
xgb_sin_lag = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.5,
    reg_lambda=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_sin_lag.fit(X_train_sin_lag, y_train)
y_pred_xgb_sin_lag = xgb_sin_lag.predict(X_test_sin_lag)

# Métricas
robustez_sin_lag = pd.DataFrame({
    "Modelo": ["Random Forest", "XGBoost"],
    "Escenario": ["Sin tasa_lag1", "Sin tasa_lag1"],
    "MAE": [
        mean_absolute_error(y_test, y_pred_rf_sin_lag),
        mean_absolute_error(y_test, y_pred_xgb_sin_lag)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_rf_sin_lag)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb_sin_lag))
    ],
    "R2": [
        r2_score(y_test, y_pred_rf_sin_lag),
        r2_score(y_test, y_pred_xgb_sin_lag)
    ]
})

robustez_sin_lag

,Modelo,Escenario,MAE,RMSE,R2
0,Random Forest,Sin tasa_lag1,686.521573,984.717823,0.329834
1,XGBoost,Sin tasa_lag1,697.753827,908.522665,0.429533


In [ ]:
# ================================
# Robustez 2 — Estabilidad temporal
# ================================

robustez_temporal = (
    rolling_resultados
    .groupby("modelo")
    .agg({
        "MAE": ["mean", "std"],
        "RMSE": ["mean", "std"],
        "R2": ["mean", "std"]
    })
    .round(3)
)

robustez_temporal

MAE             RMSE             R2       
                  mean     std     mean     std   mean    std
modelo                                                       
Random Forest  273.614  31.091  382.061  57.352  0.863  0.085
XGBoost        273.300  19.721  393.576  53.708  0.857  0.078

In [ ]:
# ================================
# Robustez 3 — Visualización estabilidad temporal
# ================================

import plotly.express as px

# Preparar dataframe
robustez_plot = robustez_temporal.copy()

robustez_plot.columns = [
    "_".join(col).strip()
    for col in robustez_plot.columns.values
]

robustez_plot = robustez_plot.reset_index()

# RMSE promedio + desviación
fig = px.bar(
    robustez_plot,
    x="modelo",
    y="RMSE_mean",
    error_y="RMSE_std",
    text="RMSE_mean",
    title="Robustez temporal — RMSE promedio y variabilidad"
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_dark",
    title_x=0.5,
    xaxis_title="Modelo",
    yaxis_title="RMSE promedio",
    showlegend=False
)

fig.show()

In [ ]:
# ================================
# Robustez 4 — Comparación con y sin tasa_lag1
# ================================

robustez_con_lag = pd.DataFrame({
    "Modelo": ["Random Forest", "XGBoost"],
    "Escenario": ["Con tasa_lag1", "Con tasa_lag1"],
    "MAE": [mae_rf, mae_xgb],
    "RMSE": [rmse_rf, rmse_xgb],
    "R2": [r2_rf, r2_xgb]
})

robustez_lag_comparacion = pd.concat(
    [robustez_con_lag, robustez_sin_lag],
    ignore_index=True
)

robustez_lag_comparacion

,Modelo,Escenario,MAE,RMSE,R2
0,Random Forest,Con tasa_lag1,269.356090,386.052101,0.896997
1,XGBoost,Con tasa_lag1,267.379886,381.712690,0.899299
2,Random Forest,Sin tasa_lag1,686.521573,984.717823,0.329834
3,XGBoost,Sin tasa_lag1,697.753827,908.522665,0.429533


In [ ]:
# ================================
# Robustez 5 — Visualización con y sin lag
# ================================

import plotly.express as px

fig = px.bar(
    robustez_lag_comparacion,
    x="Modelo",
    y="R2",
    color="Escenario",
    barmode="group",
    text="R2",
    template="plotly_dark",
    title="Impacto de eliminar tasa_lag1 sobre el desempeño"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    yaxis_title="R²",
    xaxis_title="Modelo",
    legend_title="Escenario"
)

fig.show()

In [ ]:
# ================================
# Guardar resultados de robustez
# ================================

# Guardar comparación con/sin tasa_lag1
robustez_lag_comparacion.to_csv(
    MODELS_DIR / "robustez_lag_comparacion.csv",
    index=False
)

# Aplanar columnas multiíndice
robustez_temporal_export = robustez_temporal.copy()

robustez_temporal_export.columns = [
    "_".join(col).strip()
    for col in robustez_temporal_export.columns.values
]

robustez_temporal_export = robustez_temporal_export.reset_index()

# Guardar resumen temporal
robustez_temporal_export.to_csv(
    MODELS_DIR / "robustez_temporal_resumen.csv",
    index=False
)

print("Archivos de robustez guardados correctamente.")

Archivos de robustez guardados correctamente.
